The goal of this notebook is to copy some smaller objects from previous steps into the objects directory for easier access:

- scvi model
- marker genes
- ABC CCF annotation tree
- region predictions
- region assignments

In [1]:
from pathlib import Path
import os 
import shutil 
import re 

import pandas as pd
import anndata as ad 

from dotenv import dotenv_values

In [2]:
# Setting up filepaths from config
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

OBJECTS_DIR = DATADIR / "9_objects" 

#scvi model
MODEL_IN = DATADIR / "5_clustering" / "all_control_posthoc_qc" / "models" / "scvi" / "model.pt"
MODEL_OUTDIR = OBJECTS_DIR / "scvi"
os.makedirs(MODEL_OUTDIR, exist_ok=True)
MODEL_OUT = MODEL_OUTDIR / "model.pt"

# markers
MARKERS_IN = DATADIR / "5_clustering" / 'HypoMap3D_marker_genes_qc.csv'
REG_MARKERS_DIR = DATADIR / "8_annotation" / "region_specific_markers"
MARKERS_OUTDIR = OBJECTS_DIR / "markers"
os.makedirs(MARKERS_OUTDIR, exist_ok=True)
MARKERS_OUT = MARKERS_OUTDIR / "hypomap3d_cell_type_marker_genes_all_levels.csv"

# ABC tree for spatial voxel annotations
ABC_TREE_IN = DATADIR / "7_spatial" / 'structure_tree_processed.json'
ABC_TREE_OUTDIR = OBJECTS_DIR / "spatial"
ABC_TREE_OUT = ABC_TREE_OUTDIR / "ABC_structure_tree_processed.json"

# region predictions
REGIONS_DIR = DATADIR / "7_spatial" / "analyses" 
REGIONS_ALL_IN = REGIONS_DIR / "regionwise_rm_all_wo_root.csv" 
REGIONS_ALL_FILT_IN = REGIONS_DIR / "regionwise_rm_all_wo_root_or_fibers.csv" 
REGIONS_HY_FILT_IN = REGIONS_DIR / "regionwise_rm_HY_wo_root_or_fibers.csv" 
FLAGS_IN = REGIONS_DIR / "spatial_flags.csv"
REGIONS_OUTDIR = OBJECTS_DIR / "spatial" / "regionwise_relative_masses"
REGION_ASSIGN_OUTDIR = OBJECTS_DIR / "spatial" / "region_assignments"
os.makedirs(REGIONS_OUTDIR, exist_ok=True)
os.makedirs(REGION_ASSIGN_OUTDIR, exist_ok=True)
REGIONS_ALL_OUT = REGIONS_OUTDIR / "regionwise_relative_masses.csv"
REGIONS_ALL_FILT_OUT = REGIONS_OUTDIR / "regionwise_relative_masses_wo_fibers.csv"
REGIONS_HY_FILT_OUT = REGIONS_OUTDIR / "regionwise_relative_masses_wo_fibers_or_non_HY_regions.csv"

# adata for mapping new annotations
SC_FP = OBJECTS_DIR / "singlecell" / "hypomap3d_singlecell_control.h5ad" 


### Let's copy the scvi model 

In [ ]:
shutil.copy2(MODEL_IN, MODEL_OUT)

### Let's copy the ABC structure tree

In [ ]:
shutil.copy2(ABC_TREE_IN, ABC_TREE_OUT)

PosixPath('/REDACTED-path/HypoMap3D-data/v0.9/9_objects/spatial/ABC_structure_tree_processed.json')

### Let's create ID -> annotation mappers 

In [4]:
def build_id_mapping(n_df, id_pattern, target_pattern, levels=range(1, 6),):
    mapping = {}

    # --- 1. from wide dataframe ---
    for i in levels:
        id_col = id_pattern.format(i=i)
        target_col = target_pattern.format(i=i)

        if id_col in n_df.columns and target_col in n_df.columns:
            mapping.update(
                n_df[[id_col, target_col]]
                .dropna()
                .drop_duplicates()
                .set_index(id_col)[target_col]
                .to_dict()
            )

    return mapping

In [5]:
adata = ad.read_h5ad(SC_FP, backed="r")

In [6]:
abclike_mapper = build_id_mapping(adata.obs,
    id_pattern="L{i}_ID",
    target_pattern="L{i}_abc_like")

descr_mapper = build_id_mapping(adata.obs,
    id_pattern="L{i}_ID",
    target_pattern="L{i}_descr") 

del adata

### Let's compile the markers

In [7]:
markers = pd.read_csv(MARKERS_IN)

In [8]:
markers.drop(columns="direction", inplace=True)
# get IDs
markers.cluster = markers.cluster.apply(lambda x: x.split(" ")[0])
markers.rename(columns={"cluster": "celltype_id"}, inplace=True)
# map new labels
markers["celltype_descr"] = markers.celltype_id.map(descr_mapper)
markers["celltype_abc_like"] = markers.celltype_id.map(abclike_mapper)

In [9]:
markers.head(2)

,gene_name,specificity,pctA,pctB,meanA,meanB,log2FC,p,padj,VE,rank,clustering,celltype_id,comparison,celltype_descr,celltype_abc_like
0,Foxb1,6.23657,0.106480,0.003613,0.576016,0.005636,6.674614,0.0,0.0,12.376232,22,L1,Gl1-1,OneVsSisters,Gl1-1 Glut,Gl1-1 Glut
1,Sim1,5.84080,0.230684,0.005491,0.649711,0.009302,6.125616,0.0,0.0,48.814749,8,L1,Gl1-1,OneVsSisters,Gl1-1 Glut,Gl1-1 Glut


In [10]:
# concat all region-specific markers

pattern = re.compile(r"^L\d+_markers")

files = sorted(
    f for f in REG_MARKERS_DIR.glob("*.csv")
    if pattern.match(f.stem)
)

markers_r = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

In [11]:
# get IDs
markers_r.cluster = markers_r.cluster.apply(lambda x: x.split(" ")[0])
markers_r.rename(columns={"cluster": "celltype_id", "region": "comparison"}, inplace=True)
# map new labels
markers_r["celltype_descr"] = markers_r.celltype_id.map(descr_mapper)
markers_r["celltype_abc_like"] = markers_r.celltype_id.map(abclike_mapper)

In [12]:
markers_r["clustering"] = markers_r.celltype_id.apply(lambda x: f"L{x[2]}")

In [13]:
markers_r.head(2)

,gene_name,specificity,pctA,pctB,meanA,meanB,log2FC,p,padj,VE,rank,comparison,celltype_id,celltype_descr,celltype_abc_like,clustering
0,Isl1,5.30168,0.389473,0.015569,1.120169,0.020879,5.743195,0.000000e+00,0.000000e+00,56.123360,2,AHN,Gb2-59,Gb2-59 AHN-SBPV-PVHd Pdrm12,Gb2-59 AHN-SBPV-PVHd Pdrm12 GABA,L2
1,Meis2,5.25564,0.275047,0.020495,3.007047,0.043747,6.101927,2.180353e-303,1.092357e-300,33.575891,6,AHN,Gb2-59,Gb2-59 AHN-SBPV-PVHd Pdrm12,Gb2-59 AHN-SBPV-PVHd Pdrm12 GABA,L2


In [14]:
markers_all = pd.concat([markers, markers_r], ignore_index=True)

In [15]:
# reorder
markers_all = markers_all.loc[:, [
    'clustering', 'celltype_id', 'celltype_descr', 'celltype_abc_like', 'comparison',
    'gene_name', 'specificity', 'pctA', 'pctB', 'meanA', 'meanB', 'log2FC',
    'p', 'padj', 'VE', 'rank', 
]]

In [16]:
markers_all.head(2)

,clustering,celltype_id,celltype_descr,celltype_abc_like,comparison,gene_name,specificity,pctA,pctB,meanA,meanB,log2FC,p,padj,VE,rank
0,L1,Gl1-1,Gl1-1 Glut,Gl1-1 Glut,OneVsSisters,Foxb1,6.23657,0.106480,0.003613,0.576016,0.005636,6.674614,0.0,0.0,12.376232,22
1,L1,Gl1-1,Gl1-1 Glut,Gl1-1 Glut,OneVsSisters,Sim1,5.84080,0.230684,0.005491,0.649711,0.009302,6.125616,0.0,0.0,48.814749,8


In [17]:
markers_all.to_csv(MARKERS_OUT, index=False)

In [18]:
# saving out smaller files; level-specific
for lvl in markers_all.clustering.unique():
    markers_all.loc[markers_all.clustering == lvl, :].to_csv(MARKERS_OUTDIR / f"hypomap3d_cell_type_marker_genes_{lvl}.csv", index=False)

### Let's update the regions 

In [19]:
flags = pd.read_csv(FLAGS_IN)
flags.warning = flags.warning.fillna("None")
flags_mapper = flags.set_index("cluster_id").to_dict()["warning"]

In [20]:
in_out_tuples = [
    (REGIONS_ALL_IN, REGIONS_ALL_OUT),
    (REGIONS_ALL_FILT_IN, REGIONS_ALL_FILT_OUT),
    (REGIONS_HY_FILT_IN, REGIONS_HY_FILT_OUT)
]

for in_fp, out_fp in in_out_tuples:

    reg = pd.read_csv(in_fp)

    reg = reg.rename(columns={"Unnamed: 0": "celltype_id",})
    acro_columns = list(reg.columns)[1:]
    # map new labels
    reg["celltype_descr"] = reg.celltype_id.map(descr_mapper)
    reg["celltype_abc_like"] = reg.celltype_id.map(abclike_mapper)
    reg["flag"] = reg.celltype_id.map(flags_mapper)
    # reorder 
    reg = reg.loc[:, [
        "celltype_id", "celltype_descr", "celltype_abc_like", "flag", 
    ] + acro_columns]
    reg.to_csv(out_fp, index=False)

In [21]:
reg.head(2)

,celltype_id,celltype_descr,celltype_abc_like,flag,TMv,PVa,PVH,PVHd,ADP,AHN,...,HY,PS,TMd,ME,VMPO,PeF,MMl,MMm,MMp,MMd
0,Gb1-2,Gb1-2 GABA,Gb1-2 GABA,None,0.006751,0.007684,0.006426,0.009450,0.011216,0.058012,...,0.188627,0.012184,0.002222,0.007548,0.005155,0.012813,0.000732,0.000384,0.000079,0.000775
1,Gl1-1,Gl1-1 Glut,Gl1-1 Glut,None,0.003459,0.002353,0.023333,0.006919,0.003396,0.039845,...,0.162567,0.001840,0.004350,0.003064,0.002292,0.007933,0.030843,0.024505,0.006384,0.009122


### Let's get the region assignments 

In [22]:
for lvl in range(2,6):

    pattern = re.compile(fr"^L{lvl}")

    files = sorted(
        f for f in REG_MARKERS_DIR.glob("*clusters.csv")
        if pattern.match(f.stem)
    )

    dfs = []
    for f in files:
        stem = f.stem 
        region = stem.split("_")[1]

        df = pd.read_csv(f)
        df = df.rename(columns={"cluster": "celltype_id"})
        df["celltype_descr"] = df.celltype_id.map(descr_mapper)
        df["celltype_abc_like"] = df.celltype_id.map(abclike_mapper)
        df["flag"] = df.celltype_id.map(flags_mapper)
        df["region"] = region
        dfs.append(df)
    
    concat = pd.concat(dfs, axis=0)
    concat = concat.loc[:, ["region", "celltype_id", "celltype_descr", "celltype_abc_like", 
        "region_rank", "relative_mass", "cum_mass", "flag"]]

    concat.to_csv(REGION_ASSIGN_OUTDIR / f"L{lvl}_region_assignments.csv")



/tmp/ipykernel_2127644/3746316140.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  concat = pd.concat(dfs, axis=0)
/tmp/ipykernel_2127644/3746316140.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  concat = pd.concat(dfs, axis=0)
/tmp/ipykernel_2127644/3746316140.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the

In [23]:
concat.head(2)

,region,celltype_id,celltype_descr,celltype_abc_like,region_rank,relative_mass,cum_mass,flag
0,ADP,Gb5-449,Gb5-449 MPO Penk.Arap2,Gb5-449 SI-MPO-LPO Lhx8 GABA_3-4-1,2,0.114179,0.218961,Signal
1,ADP,Gb5-571,Gb5-571 MPO Zfp536.Foxp2,Gb5-571 MPO-ADP Lhx8 GABA_2-1-1,3,0.107814,0.389221,None
